# 07 · Drift dữ liệu, drift mô hình và chiến lược bảo trì (phụ trách: Chưởng)

**Đề:** *"Detect data drift and model drift"* (có trong tên đề: *"Drift Monitoring"*), prototype *"evaluated by … drift metrics"*.
Mã nguồn: `trustworthy_ai/drift.py`, `backend/app/routers/system.py` (`POST /drift/{task}`).

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

In [ ]:
R = load_results(); dr = R['drift']; print('Tham chiếu:', dr['reference_period'], '| ngưỡng null:', dr['null_thresholds'])

## 1. Bảng drift metrics chính thức theo lô

In [ ]:
pd.DataFrame([{"period": r["period"], "n": r["n"], "tỷ lệ dương": r["positive_rate"], "O/E": r["oe_ratio"],
               "% PSI>0.25": r["share_features_major"], "vượt null": r.get("n_features_over_null"),
               "AUC miền": (r.get("domain_classifier") or {}).get("auc"), "PSI điểm": r["score_psi"], "trạng thái": r["status"]}
              for r in dr["periods"]])

## 2. Drift mô hình: huấn luyện giai đoạn cũ → đánh giá giai đoạn mới

In [ ]:
pd.DataFrame(R['temporal_validation']['rows'])

## 3. Ba chiến lược bảo trì

In [ ]:
mt = pd.DataFrame(R['maintenance']['rows']); mt.pivot_table(index='period', columns='strategy', values=['oe_ratio', 'pr_auc'])

## 4. Drift mã hóa ICD-9 → ICD-10

In [ ]:
R['coding_drift'].get('largest_version_gaps'), R['coding_drift'].get('icd_version_mix_by_period')

## 5. Giám sát một lô mới (không cần nhãn) qua gói mô hình

In [ ]:
from trustworthy_ai.bundle import HeartGuardModel
from backend.app.demo_cases import demo_cases
m = HeartGuardModel(sorted((ROOT / "artifacts/models" / TASK).glob("*/SHA256SUMS"))[-1].parent)
batch = [dict(c["features"]) for c in demo_cases(TASK)] * 15
for b in batch:
    for k in list(b):
        if k.startswith("creatinine_") and b[k] is not None and not k.endswith("_missing"): b[k] *= 1.5
rep = m.drift_report(batch); print(rep["status"], "| PSI điểm:", rep.get("score_psi")); pd.DataFrame(rep["features"]).head(8)

## Câu hỏi tự kiểm tra
1. Drift dữ liệu khác drift mô hình thế nào? Cái nào đo được ngay, cái nào cần chờ nhãn (P2: 30 ngày, P3: 90 ngày)?
2. Vì sao ngưỡng PSI 0.25 quy ước chưa đủ? Ngưỡng null tính thế nào?
3. AUC của bộ phân loại miền 0.5 và 0.8 nghĩa là gì?
4. Khi tỷ lệ tử vong tăng từ 3.79% lên 8.01%, chiến lược nào khôi phục O/E rẻ nhất?